# Ingestão da camada silver

Neste notebook iremos recuperar os dados ingeridos na camada bronze e realizar análises de Data Quality e algumas transformações para ingerir na camada silver

Análise de Qualidade de Dados - Tabelas Bronze (dim_* e fact_*)
- Identificar possíveis problemas: nulos, chaves duplicadas, integridade referencial, valores inválidos e domínios

In [0]:
%python
import importlib.util

spec = importlib.util.spec_from_file_location(
    "data_quality",
    "/Workspace/Users/bueno1982@gmail.com/data-quality-challenge/utils/data_quality.py",
)
data_quality = importlib.util.module_from_spec(spec)
spec.loader.exec_module(data_quality)
data_quality.spark = spark  # injeta a SparkSession do notebook no escopo do módulo
DataQuality = data_quality.DataQuality

CATALOG = "dataquality_challenge"
SCHEMA = "bronze"

TABLES = [
    "dim_calendario",
    "dim_loja",
    "dim_produto",
    "fact_market_share_provider_a",
    "fact_market_share_provider_b",
    "coverage_provider",
    "customer_territory_history",
]

RULES_PATH = "/Workspace/Users/bueno1982@gmail.com/data-quality-challenge/dq_rules/bronze.json"

In [0]:
%python
# Validação de Data Quality baseada em regras (bronze.json)

all_results = []
for TABLE in TABLES:
    dq = DataQuality(CATALOG, SCHEMA, TABLE, RULES_PATH)
    results = dq.validate()
    all_results.extend(results)

# Resumo
failed = [r for r in all_results if r["status"] == "FAIL"]
print(f"ANÁLISE DE QUALIDADE DE DADOS — {len(failed)} de {len(all_results)} regras reprovaram\n")

df_results = spark.createDataFrame(all_results).\
    select("rule_id", "severity", "status", "classification", "table", "column", "check",
           "total_count", "fail_count", "percent", "max_percent", "treatment").\
    orderBy("status", "severity", "table", "percent", ascending=[True, True, True, False])

df_results.display()

## Camada Silver — Tratamento & Quarentena

**Estratégia baseada nos resultados do DQ da bronze:**

- **Regra falhou (percent > max_percent):** aplicar tratamento automático
  - Nulos em `territory_id` e `seller_id` (dim_loja) -> `'UNKNOWN'`
  - EANs duplicados (dim_produto) -> manter primeiro registro

- **Regra passou com percent > 0 (0 < percent <= max_percent):** enviar registros problemáticos para quarentena
  - `state` nulo (dim_loja)
  - `category` nulo (dim_produto)
  - Valores negativos, órfãos de RI e chaves duplicadas (fatos)

- **Regra passou com percent = 0:** nenhuma ação necessária

**Rastreabilidade:**
- `_dq_action` — tratamento aplicado por registro (silver)
- `_processed_at` — timestamp de processamento (silver)
- `_quarantine_reason` — motivo da quarentena (quarentena)
- `_quarantined_at` — timestamp de quarentena (quarentena)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS dataquality_challenge.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS dataquality_challenge.quarantine")

# Remover tabelas de quarentena antigas (nomenclatura anterior)
for old_t in ["dim_produto_duplicates", "fact_ms_a_orphans", "fact_ms_b_orphans"]:
    spark.sql(f"DROP TABLE IF EXISTS dataquality_challenge.quarantine.{old_t}")

# dim_calendario — sem problemas, pass-through
spark.sql("""
CREATE OR REPLACE TABLE dataquality_challenge.silver.dim_calendario AS
SELECT
  date, year, month, week, year_week, week_start_date, week_end_date,
  'clean' AS _dq_action,
  CURRENT_TIMESTAMP() AS _processed_at
FROM dataquality_challenge.bronze.dim_calendario
""")
print(f"dim_calendario silver: {spark.table('dataquality_challenge.silver.dim_calendario').count()} registros")

In [0]:
# dim_loja — auto-fix nulos (territory_id, seller_id: FAIL), quarentena state nulo (PASS com percent > 0)

# Quarentena: registros com state nulo
spark.sql("""
CREATE OR REPLACE TABLE dataquality_challenge.quarantine.dim_loja_issues AS
SELECT
  store_id, store_name, cnpj, retailer_name, channel, city, state,
  territory_id, seller_id, latitude, longitude, active_flag,
  'null_state' AS _quarantine_reason,
  CURRENT_TIMESTAMP() AS _quarantined_at
FROM dataquality_challenge.bronze.dim_loja
WHERE state IS NULL
""")

# Silver: auto-fix territory_id e seller_id (FAIL), excluir registros em quarentena
spark.sql("""
CREATE OR REPLACE TABLE dataquality_challenge.silver.dim_loja AS
SELECT
  store_id, store_name, cnpj, retailer_name, channel, city, state,
  COALESCE(territory_id, 'UNKNOWN') AS territory_id,
  COALESCE(seller_id, 'UNKNOWN') AS seller_id,
  latitude, longitude, active_flag,
  CASE
    WHEN territory_id IS NULL OR seller_id IS NULL THEN 'filled_null'
    ELSE 'clean'
  END AS _dq_action,
  CURRENT_TIMESTAMP() AS _processed_at
FROM dataquality_challenge.bronze.dim_loja
WHERE state IS NOT NULL
""")
print(f"dim_loja silver: {spark.table('dataquality_challenge.silver.dim_loja').count()} registros")
print(f"dim_loja quarentena: {spark.table('dataquality_challenge.quarantine.dim_loja_issues').count()} registros")

In [0]:
# dim_produto — auto-fix EANs duplicados (FAIL), quarentena category nulo (PASS com percent > 0)

# Quarentena: EANs duplicados (auto-fix) + registros com category nulo
spark.sql("""
CREATE OR REPLACE TABLE dataquality_challenge.quarantine.dim_produto_issues AS
WITH ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY ean ORDER BY product_id) AS _rn
  FROM dataquality_challenge.bronze.dim_produto
)
SELECT
  product_id, ean, product_description, brand, manufacturer, category,
  subcategory, package_size, package_unit, is_own_product,
  valid_from, valid_to, source_system,
  CASE
    WHEN _rn > 1 THEN 'duplicate_ean'
    WHEN category IS NULL THEN 'null_category'
  END AS _quarantine_reason,
  CURRENT_TIMESTAMP() AS _quarantined_at
FROM ranked
WHERE _rn > 1 OR category IS NULL
""")

# Silver: manter primeiro EAN (auto-fix), excluir registros com category nulo (quarentena)
spark.sql("""
CREATE OR REPLACE TABLE dataquality_challenge.silver.dim_produto AS
SELECT
  product_id, ean, product_description, brand, manufacturer, category,
  subcategory, package_size, package_unit, is_own_product,
  valid_from, valid_to, source_system,
  'clean' AS _dq_action,
  CURRENT_TIMESTAMP() AS _processed_at
FROM (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY ean ORDER BY product_id) AS _rn
  FROM dataquality_challenge.bronze.dim_produto
)
WHERE (_rn = 1 OR ean IS NULL)
  AND category IS NOT NULL
""")
print(f"dim_produto silver: {spark.table('dataquality_challenge.silver.dim_produto').count()} registros")
print(f"dim_produto quarentena: {spark.table('dataquality_challenge.quarantine.dim_produto_issues').count()} registros")

In [0]:
# fact_market_share_provider_a — quarentena negativos, órfãos (RI) e duplicados (PASS com percent > 0)

# Quarentena: registros com valores negativos, RI órfãos ou chaves duplicadas
spark.sql("""
CREATE OR REPLACE TABLE dataquality_challenge.quarantine.fact_ms_a_issues AS
WITH ranked AS (
  SELECT
    f.*,
    ROW_NUMBER() OVER (PARTITION BY f.week, f.store_id, f.ean ORDER BY f.ingestion_timestamp DESC) AS _rn,
    d.store_id AS _matched_store,
    p.ean AS _matched_product
  FROM dataquality_challenge.bronze.fact_market_share_provider_a f
  LEFT JOIN dataquality_challenge.silver.dim_loja d ON f.store_id = d.store_id
  LEFT JOIN dataquality_challenge.silver.dim_produto p ON f.ean = p.ean
)
SELECT
  week, store_id, ean, sales_volume, sales_value, units, average_price,
  source_file, ingestion_timestamp, sales_value_brl, sold_volume,
  CASE
    WHEN sales_value < 0 OR sales_value_brl < 0 THEN 'negative_value'
    WHEN _matched_store IS NULL THEN 'orphan_store'
    WHEN _matched_product IS NULL THEN 'orphan_product'
    WHEN _rn > 1 THEN 'duplicate_key'
  END AS _quarantine_reason,
  CURRENT_TIMESTAMP() AS _quarantined_at
FROM ranked
WHERE sales_value < 0 OR sales_value_brl < 0
   OR _matched_store IS NULL OR _matched_product IS NULL
   OR _rn > 1
""")

# Silver: apenas registros limpos (sem negativos, RI válida, sem duplicados)
spark.sql("""
CREATE OR REPLACE TABLE dataquality_challenge.silver.fact_market_share_provider_a AS
SELECT
  week, store_id, ean, sales_volume, sales_value, units, average_price,
  source_file, ingestion_timestamp, sales_value_brl, sold_volume,
  'clean' AS _dq_action,
  CURRENT_TIMESTAMP() AS _processed_at
FROM (
  SELECT
    f.*,
    ROW_NUMBER() OVER (PARTITION BY f.week, f.store_id, f.ean ORDER BY f.ingestion_timestamp DESC) AS _rn
  FROM dataquality_challenge.bronze.fact_market_share_provider_a f
  INNER JOIN dataquality_challenge.silver.dim_loja d ON f.store_id = d.store_id
  INNER JOIN dataquality_challenge.silver.dim_produto p ON f.ean = p.ean
  WHERE f.sales_value >= 0 AND f.sales_value_brl >= 0
)
WHERE _rn = 1
""")
print(f"fact_a silver: {spark.table('dataquality_challenge.silver.fact_market_share_provider_a').count()} registros")
print(f"fact_a quarentena: {spark.table('dataquality_challenge.quarantine.fact_ms_a_issues').count()} registros")

In [0]:
# fact_market_share_provider_b — quarentena negativos, órfãos (RI) e duplicados (PASS com percent > 0)

# Quarentena: registros com valores negativos, RI órfãos ou chaves duplicadas
spark.sql("""
CREATE OR REPLACE TABLE dataquality_challenge.quarantine.fact_ms_b_issues AS
WITH ranked AS (
  SELECT
    f.*,
    ROW_NUMBER() OVER (PARTITION BY f.reference_week, f.customer_code, f.product_ean ORDER BY f.load_date DESC) AS _rn,
    d.store_id AS _matched_store,
    p.ean AS _matched_product
  FROM dataquality_challenge.bronze.fact_market_share_provider_b f
  LEFT JOIN dataquality_challenge.silver.dim_loja d ON f.customer_code = d.store_id
  LEFT JOIN dataquality_challenge.silver.dim_produto p ON f.product_ean = p.ean
)
SELECT
  reference_week, customer_code, product_ean, qty_kg, revenue,
  unit_count, competitor_flag, load_date, market, source_file,
  sales_value_brl, sold_volume,
  CASE
    WHEN revenue < 0 OR sales_value_brl < 0 THEN 'negative_value'
    WHEN _matched_store IS NULL THEN 'orphan_store'
    WHEN _matched_product IS NULL THEN 'orphan_product'
    WHEN _rn > 1 THEN 'duplicate_key'
  END AS _quarantine_reason,
  CURRENT_TIMESTAMP() AS _quarantined_at
FROM ranked
WHERE revenue < 0 OR sales_value_brl < 0
   OR _matched_store IS NULL OR _matched_product IS NULL
   OR _rn > 1
""")

# Silver: apenas registros limpos (sem negativos, RI válida, sem duplicados)
spark.sql("""
CREATE OR REPLACE TABLE dataquality_challenge.silver.fact_market_share_provider_b AS
SELECT
  reference_week, customer_code, product_ean, qty_kg, revenue,
  unit_count, competitor_flag, load_date, market, source_file,
  sales_value_brl, sold_volume,
  'clean' AS _dq_action,
  CURRENT_TIMESTAMP() AS _processed_at
FROM (
  SELECT
    f.*,
    ROW_NUMBER() OVER (PARTITION BY f.reference_week, f.customer_code, f.product_ean ORDER BY f.load_date DESC) AS _rn
  FROM dataquality_challenge.bronze.fact_market_share_provider_b f
  INNER JOIN dataquality_challenge.silver.dim_loja d ON f.customer_code = d.store_id
  INNER JOIN dataquality_challenge.silver.dim_produto p ON f.product_ean = p.ean
  WHERE f.revenue >= 0 AND f.sales_value_brl >= 0
)
WHERE _rn = 1
""")
print(f"fact_b silver: {spark.table('dataquality_challenge.silver.fact_market_share_provider_b').count()} registros")
print(f"fact_b quarentena: {spark.table('dataquality_challenge.quarantine.fact_ms_b_issues').count()} registros")

In [0]:
# coverage_provider — DQ + pass-through (sem tratamento automático necessário)
spark.sql("""
CREATE OR REPLACE TABLE dataquality_challenge.silver.coverage_provider AS
SELECT
  provider, year_week, retailer_name,
  expected_stores, received_stores, file_received,
  expected_delivery_date, actual_delivery_date,
  'clean' AS _dq_action,
  CURRENT_TIMESTAMP() AS _processed_at
FROM dataquality_challenge.bronze.coverage_provider
WHERE expected_stores >= 0
  AND received_stores >= 0
  AND file_received IN ('Y', 'N')
""")

# Quarentena: registros com valores inválidos (se houver)
spark.sql("""
CREATE OR REPLACE TABLE dataquality_challenge.quarantine.coverage_provider_issues AS
SELECT
  provider, year_week, retailer_name,
  expected_stores, received_stores, file_received,
  expected_delivery_date, actual_delivery_date,
  CASE
    WHEN expected_stores < 0 THEN 'negative_expected_stores'
    WHEN received_stores < 0 THEN 'negative_received_stores'
    WHEN file_received NOT IN ('Y', 'N') THEN 'invalid_file_received'
  END AS _quarantine_reason,
  CURRENT_TIMESTAMP() AS _quarantined_at
FROM dataquality_challenge.bronze.coverage_provider
WHERE expected_stores < 0
   OR received_stores < 0
   OR file_received NOT IN ('Y', 'N')
""")
print(f"coverage_provider silver: {spark.table('dataquality_challenge.silver.coverage_provider').count()} registros")
print(f"coverage_provider quarentena: {spark.table('dataquality_challenge.quarantine.coverage_provider_issues').count()} registros")

In [0]:
# customer_territory_history — SCD Type 2: dedup por chave composta (store_id + category + valid_from)
# Auto-fix: nulos em territory_id e seller_id (se houver FAIL)
spark.sql("""
CREATE OR REPLACE TABLE dataquality_challenge.silver.customer_territory_history AS
SELECT
  store_id, category,
  COALESCE(territory_id, 'UNKNOWN') AS territory_id,
  COALESCE(seller_id, 'UNKNOWN') AS seller_id,
  valid_from, valid_to, assignment_source,
  CASE
    WHEN territory_id IS NULL OR seller_id IS NULL THEN 'filled_null'
    ELSE 'clean'
  END AS _dq_action,
  CURRENT_TIMESTAMP() AS _processed_at
FROM (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY store_id, category, valid_from
      ORDER BY valid_to DESC
    ) AS _rn
  FROM dataquality_challenge.bronze.customer_territory_history
)
WHERE _rn = 1
  AND valid_from <= valid_to
""")
print(f"customer_territory_history silver: {spark.table('dataquality_challenge.silver.customer_territory_history').count()} registros")

In [0]:
# sensitive_store_contacts — estratégia de minimização e proteção de dados sensíveis (Seção 10)
# Mascaramento de email e telefone; apenas store_id e retailer_name são mantidos sem mascaramento
spark.sql("""
CREATE OR REPLACE TABLE dataquality_challenge.silver.sensitive_store_contacts AS
SELECT
  store_id,
  contact_name,
  CONCAT(SUBSTRING(contact_email, 1, 2), '***@***') AS contact_email_masked,
  CONCAT('***', SUBSTRING(contact_phone, -4)) AS contact_phone_masked,
  retailer_name,
  'masked' AS _dq_action,
  CURRENT_TIMESTAMP() AS _processed_at
FROM dataquality_challenge.bronze.sensitive_store_contacts
""")
print(f"sensitive_store_contacts silver (mascarado): {spark.table('dataquality_challenge.silver.sensitive_store_contacts').count()} registros")

In [0]:
# Fato consolidado — UNION dos dois providers com schema padronizado (Seção 9: evidências de Union)
# Provider A: week, store_id, ean, sales_value_brl, sold_volume
# Provider B: reference_week, customer_code, product_ean, sales_value_brl, sold_volume
# Schema padronizado: year_week, store_id, ean, sales_value_brl, sold_volume, provider

spark.sql("""
CREATE OR REPLACE TABLE dataquality_challenge.silver.fact_market_share_consolidated AS
-- Provider A
SELECT
  week AS year_week,
  store_id,
  ean,
  sales_value_brl,
  sold_volume,
  'A' AS provider,
  source_file,
  ingestion_timestamp,
  'provider_a' AS source_table,
  'clean' AS _dq_action,
  CURRENT_TIMESTAMP() AS _processed_at
FROM dataquality_challenge.silver.fact_market_share_provider_a

UNION ALL

-- Provider B
SELECT
  reference_week AS year_week,
  customer_code AS store_id,
  product_ean AS ean,
  sales_value_brl,
  sold_volume,
  'B' AS provider,
  source_file,
  load_date AS ingestion_timestamp,
  'provider_b' AS source_table,
  'clean' AS _dq_action,
  CURRENT_TIMESTAMP() AS _processed_at
FROM dataquality_challenge.silver.fact_market_share_provider_b
""")
print(f"fact_market_share_consolidated: {spark.table('dataquality_challenge.silver.fact_market_share_consolidated').count()} registros")
print(f"  Provider A: {spark.table('dataquality_challenge.silver.fact_market_share_consolidated').filter('provider = "A"').count()}")
print(f"  Provider B: {spark.table('dataquality_challenge.silver.fact_market_share_consolidated').filter('provider = "B"').count()}")

In [0]:
# Resumo: bronze vs silver vs quarentena
print("=== RESUMO DA CAMADA SILVER ===\n")
for t in TABLES:
    bronze_count = spark.table(f"dataquality_challenge.bronze.{t}").count()
    silver_count = spark.table(f"dataquality_challenge.silver.{t}").count()
    diff = bronze_count - silver_count
    print(f"{t}: bronze={bronze_count} -> silver={silver_count} (removidos: {diff})")

In [0]:
print("\n=== QUARENTENA ===")
for row in spark.sql("SHOW TABLES IN dataquality_challenge.quarantine").collect():
    qname = f"dataquality_challenge.quarantine.{row.tableName}"
    print(f"  {row.tableName}: {spark.table(qname).count()} registros")

In [0]:
# Validação DQ na camada silver
print("\n=== VALIDAÇÃO DQ NA CAMADA SILVER ===\n")
all_silver_results = []
for table in TABLES:
    dq = DataQuality("dataquality_challenge", "silver", table, RULES_PATH)
    results = dq.validate()
    all_silver_results.extend(results)

failed = [r for r in all_silver_results if r["status"] == "FAIL"]
print(f"DQ SILVER — {len(failed)} de {len(all_silver_results)} regras reprovaram\n")

df_silver_dq = spark.createDataFrame(all_silver_results).\
    select("rule_id", "severity", "status", "classification", "table", "column", "check",
           "total_count", "fail_count", "percent", "max_percent", "treatment").\
    orderBy("status", "severity", "table", "percent", ascending=[True, True, True, False])

# Persistir resultados DQ como tabela para auditoria (Seção 7: "Log detalhado e métricas agregadas")
(df_silver_dq.write.mode("overwrite")
    .saveAsTable("dataquality_challenge.silver.dq_results"))
print(f"Resultados DQ persistidos: {df_silver_dq.count()} registros em silver.dq_results")

df_silver_dq.display()